In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
path = os.getcwd()
drive_path = 'drive/MyDrive/RAH/'

In [ ]:
os.system(f'unzip {drive_path}musan_small.zip -d {path}')
os.system(f'unzip {drive_path}data3.zip -d {path}')
os.system(f'unzip {drive_path}rirs_noises_small.zip -d {path}')

0

In [ ]:
import torch
import torch, torchaudio, glob
import random
import scipy.signal
import numpy as np
from heapq import nlargest
import editdistance

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
seed_everything(33)

# Transformer

In [ ]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model=512, d_ff=1024, dropout=0.1, **kwargs):
        super().__init__()
        self.ff = torch.nn.Sequential(
            torch.nn.LayerNorm(d_model),
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.ff(x)

class SelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

    def forward(self, x):
        x = self.norm(x)
        b = x.shape[0]
        q = self.q_linear(x).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).view(b, -1, self.n_heads, self.d_head)


        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale

        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)

        out = self.out(out)
        return out

class Encoder(torch.nn.Module):
    def __init__(self, nb_layers=6, seq_len=400, **kwargs):
        super().__init__()
        self.pos = torch.nn.Parameter(torch.randn(1, seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([SelfAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])

    def forward(self, x):
        b, t, d = x.shape
        x = x + self.pos[:, :t, :]
        for att, ff in zip(self.att, self.ff):
            x = x + att(x)
            x = x + ff(x)
        return x


class CausalSelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, seq_len=400, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

        self.register_buffer("mask", torch.tril(torch.ones(seq_len, seq_len)).view(1, 1, seq_len, seq_len) == 0)


    def forward(self, x):
        x = self.norm(x)
        b, n, d = x.shape
        q = self.q_linear(x).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).view(b, -1, self.n_heads, self.d_head)

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale

        scores = scores.masked_fill(self.mask[:,:,:n,:n], float('-inf'))
        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out

class CrossAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

    def forward(self, x1, x2):
        x1 = self.norm1(x1)
        x2 = self.norm2(x2)
        b = x1.shape[0]
        q = self.q_linear(x1).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x2).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x2).view(b, -1, self.n_heads, self.d_head)

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale

        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)

        out = self.out(out)
        return out, att

class Decoder(torch.nn.Module):
    def __init__(self, nb_layers=6, seq_len=400, **kwargs):
        super().__init__()
        self.pos = torch.nn.Parameter(torch.randn(1, seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([CausalSelfAttention(**kwargs) for _ in range(nb_layers)])
        self.cross_att = torch.nn.ModuleList([CrossAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])

    def forward(self, x, enc):
        b, t, d = x.shape
        x = x + self.pos[:, :t, :]
        for att, cross_att, ff in zip(self.att, self.cross_att, self.ff):
            x = x + att(x)
            x = x + cross_att(x, enc)[0]
            x = x + ff(x)
        return x

class SpecAug(torch.nn.Module):
    def __init__(self, prob_t_warp=0.5,
                       t_factor=(0.9, 1.1),
                       f_mask_width = (0, 8),
                       t_mask_width = (0, 10),
                       nb_f_masks=[1,2],
                       nb_t_masks=[1,2],
                       ):
        super().__init__()
        self.t_factor = t_factor
        self.f_mask_width = f_mask_width
        self.t_mask_width = t_mask_width
        self.nb_f_masks = nb_f_masks
        self.nb_t_masks = nb_t_masks
        self.prob_t_warp = prob_t_warp

    def time_warp(self, x):
        x = torch.nn.functional.interpolate(x, size=(int(x.shape[2]*np.random.uniform(*self.t_factor)), ))
        return x

    def freq_mask(self, x):
        for _ in range(np.random.randint(*self.nb_f_masks)):
            f = np.random.randint(*self.f_mask_width)
            f0 = np.random.randint(0, x.shape[1]-f)
            x[:,f0:f0+f,:] = 0
        return x

    def time_mask(self, x):
        for _ in range(np.random.randint(*self.nb_t_masks)):
            t = np.random.randint(*self.t_mask_width)
            t0 = np.random.randint(0, x.shape[2]-t)
            x[:,:,t0:t0+t] = 0
        return x

    def forward(self, x):
        # Moo: Time wark quitado por influir negativamente en la performance
        # if np.random.uniform() < self.prob_t_warp:
        #     x = self.time_warp(x)
        x = self.freq_mask(x)
        x = self.time_mask(x)
        return x

# Feature Extractor

In [ ]:
#actividad extra 2
class PretrainedFeatures(torch.nn.Module):
    def __init__(self, freeze=True, d_model=512, **kwargs):
        super().__init__()
        from transformers import WavLMModel
        self.fe = WavLMModel.from_pretrained("patrickvonplaten/wavlm-libri-clean-100h-base-plus")
        self.spec_aug = SpecAug()
        self.linear = torch.nn.Linear(768, d_model)
        if freeze:
            for p in self.fe.parameters():
                p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            x = self.fe(x).last_hidden_state
        # if self.training:
        #     x = self.spec_aug(x.transpose(1,2)).transpose(1,2)
        x = self.linear(x)
        return x

class AudioFeatures(torch.nn.Module):
    def __init__(self, feat_dim=80, d_model=512, **kwargs):
        super().__init__()
        self.fe = torchaudio.transforms.MelSpectrogram(
                        n_fft=512,
                        win_length=25*16,
                        hop_length=10*16,
                        n_mels=feat_dim)                            # 25ms window, 10ms shift
        self.spec_aug = SpecAug()
        self.linear = torch.nn.Linear(feat_dim, d_model)

    def forward(self, x):
        x = self.fe(x)
        x = (x+1e-6).log()
        if self.training:
            x = self.spec_aug(x)
        x = x.transpose(1, 2)
        x = self.linear(x)
        return x

class AudioTransformer(torch.nn.Module):
    def __init__(self, vocab_size=24, **kwargs):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = kwargs['seq_len']

        #actividad extra 2
        # nb_layers = kwargs['nb_layers']
        # self.fe = PretrainedFeatures(**kwargs)
        self.fe = AudioFeatures(**kwargs)
        # kwargs['nb_layers'] = 1
        self.enc = Encoder(**kwargs)
        # kwargs['nb_layers'] = nb_layers
        self.emb = torch.nn.Embedding(vocab_size, kwargs['d_model'])
        self.dec = Decoder(**kwargs)
        self.out = torch.nn.Linear(kwargs['d_model'], vocab_size)

        self.n_beams = kwargs['n_beams']


    def encoder(self, x):
        x = self.fe(x)
        return self.enc(x)

    def decoder(self, y, enc):
        y = self.emb(y)
        dec = self.dec(y, enc)
        return self.out(dec)

    def forward(self, x, y):
        enc = self.encoder(x)
        return self.decoder(y, enc)

    def loss(self, x, y):
        logits = self(x, y[:,:-1])
        target = y[:,1:]
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, self.vocab_size),
                                                 target.reshape(-1))
        return loss

    def seq_finished(self, seq):
      return seq[-1] == 22

    def all_seq_finished(self, seqs):
        return sum([self.seq_finished(seq) for (seq, _) in seqs]) == len(seqs)

    def all_seq_in_range(self, seqs):
        return sum([len(seq) < 25 for (seq, _) in seqs]) == len(seqs)

    # Activity 3 beam search
    def beam_search_gen(self, start, enc, device):
        hypotheses_seq = [[start, 1]]

        while not self.all_seq_finished(hypotheses_seq) and self.all_seq_in_range(hypotheses_seq):
            new_hypotheses_seq = []

            for i, (seq, p) in enumerate(hypotheses_seq):
                if not self.seq_finished(seq):
                    logits = self.decoder(torch.tensor(seq).unsqueeze(0).to(device), enc)
                    logits = logits[:,-1]

                    probs = torch.nn.functional.softmax(logits[-1])
                    _, indexes = torch.topk(probs, self.n_beams)
                    for j in indexes:
                        aux_seq = seq.copy()
                        p_aux = p * probs[j]
                        aux_seq.append(j.item())

                        new_hypotheses_seq.append([aux_seq, p_aux])
                else:
                    if not len(seq) == 2: # [20, 22]
                        new_hypotheses_seq.append([seq.copy(), p])
            sorted_hyp = sorted(new_hypotheses_seq, key=lambda element: element[1], reverse=True)
            hypotheses_seq = sorted_hyp[:self.n_beams].copy()

        return hypotheses_seq[0][0]


    def generate(self, x):
        device = next(self.parameters()).device
        self.eval()
        x = torch.nn.functional.pad(torch.tensor(x), (0, self.seq_len-len(x)), value=23)
        if len(x.shape) == 1:
            x = x[None, :]
        y = [20,]
        with torch.no_grad():
            enc = self.encoder(x.to(device))

            #Actividad extra 3
            y_h = self.beam_search_gen(y, enc, device)

            # while y[-1] != 22 and len(y) < self.seq_len:
            #     logits = self.decoder(torch.tensor(y).unsqueeze(0).to(device), enc)
            #     y.append(logits.argmax(-1)[:,-1].item())

        return y_h


# Dataset

In [ ]:
class NoiseAug(object):
    def __init__(self, noise_dir='musan_small/', prob=0.5):
        noise_dir = os.getcwd()+'/musan_small/'
        print(noise_dir)
        self.prob = prob
        self.noises = glob.glob(noise_dir+'/*/*.wav')

    def __call__(self, x):
        if np.random.uniform() < self.prob:
            n = torchaudio.load( np.random.choice(self.noises) )[0][0]
            if len(n) < len(x):
                n = torch.nn.functional.pad(n, (0, len(x)-len(n)), value=0)
            elif len(n) > len(x):
                t0 = np.random.randint(0, len(n) - len(x))
                n = n[t0:t0+len(x)]
            n = n.numpy()
            p_x = x.std()**2
            p_n = n.std()**2
            snr = np.random.uniform(5, 15)
            n = n * np.sqrt(p_x/p_n) * np.power(10, -snr/20)
            x = x + n
        return x

class RIRAug(object):
    def __init__(self, rir_dir='RIRS_NOISES_small/simulated_rirs_small/', prob=0.5):
        rir_dir = os.getcwd() + '/RIRS_NOISES_small/simulated_rirs_small/'
        self.prob = prob
        self.rirs = glob.glob(rir_dir+'/*.wav')

    def __call__(self, x):
        if np.random.uniform() < self.prob:
            n = len(x)
            rir = torchaudio.load( np.random.choice(self.rirs) )[0][0]
            rir = rir.numpy()
            rir = rir / np.max(np.abs(rir))
            x = scipy.signal.convolve(x, rir)
            t0 = np.argmax(np.abs(rir))
            x = x[t0:t0+n]
        return x

def identity(x):
    return x

def num2list(n):
    return [int(d) for d in str(n)]

def get_ground_truth(label):
    num_list_representation = num2list(label)
    num_sum = num2list( sum(num_list_representation))
    if len(num_list_representation) == 1:
        y = [20, ] + num_list_representation + [19, ] + num_list_representation + [21, ] + num_sum + [22,]
    elif len(num_list_representation) == 2:
        y = [20, ] + num_list_representation + [19, ] + num2list( sum(num_list_representation) ) + [21, ] + num_sum + [22,]
    elif len(num_list_representation) == 3:
        y = [20, ] + num_list_representation + [19, ] + num2list( sum(num_list_representation[0:2]) ) +  [18] + [num_list_representation[2]] + [21, ] + num_sum + [22,]
    else:
        y = [20, ] + num_list_representation + [19, ] + num2list( sum(num_list_representation[0:2]) ) +  [18] + num2list( sum(num_list_representation[2:]) ) + [21, ] + num_sum + [22,]
    return y

class TrainDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir='data3/train', audio_len=4*16000, transform=[identity], seq_len=15):
        self.data_dir = os.getcwd() + '/data3/train'
        self.transform = transform
        self.audio_len = audio_len
        self.seq_len = seq_len
        self.files = sorted( glob.glob(data_dir+'/*.wav') )
        print(len(self.files))

    def __len__(self):
        return len(self.files)


    def __getitem__(self, idx):
        x, fs = torchaudio.load(self.files[idx])

        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]

        x = x[0].numpy()
        for t in self.transform:
            x = t(x)

        label = self.files[idx].split('.')[-2].split('_')[-1]
        # print(x.shape, x.dtype)
        label = label.replace('o', '0')

        y = get_ground_truth(label)

        y = torch.nn.functional.pad(torch.tensor(y), (0, self.seq_len-len(y)), value=23)
        return x, y


class TestDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir='data3/test', audio_len=4*16000, seq_len=15):
        sata_dir = os.getcwd() + '/data3/test'
        self.audio_len = audio_len
        self.seq_len = seq_len
        self.files = sorted(glob.glob(data_dir+'/*.wav'))
        print(len(self.files))

    def __len__(self):
        return len(self.files)


    def __getitem__(self, idx):
        x, fs = torchaudio.load(self.files[idx])
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]

        x = x[0]
        label = self.files[idx].split('.')[-2].split('_')[-1]
        # print(x.shape, x.dtype)
        label = label.replace('o', '0')

        y = get_ground_truth(label)

        y = torch.nn.functional.pad(torch.tensor(y), (0, self.seq_len-len(y)), value=23)
        return x, y

trainset = TrainDataset(transform=[NoiseAug(), RIRAug()])
# trainset = TrainDataset()

testset = TestDataset()
eval_size = int(0.3 * len(testset))
test_size = len(testset) - eval_size
evalset, _testset = torch.utils.data.random_split(testset, [eval_size, test_size])


/content/musan_small/
8000
2000


# Train the network

In [ ]:
model = AudioTransformer(vocab_size=24, d_model=256, nb_layers=4, n_beams=3, flash=False, # 4 layers
                          d_ff=512, n_heads=8, d_head=32, dropout=0.1, seq_len=500)

In [ ]:
if os.path.exists(drive_path+'model_mix.pt'):
    model.load_state_dict(torch.load(drive_path+'model_mix.pt'))
    print('weights loaded')

device = 'cuda'
model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

nb_epochs = 22
batch_size = 128

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
min_err = 1000
for e in range(nb_epochs):
    loss_sum = 0
    for num_list_representation, label in trainloader:
        num_list_representation = num_list_representation.to(device)
        label = label.to(device)
        opt.zero_grad()
        loss = model.loss(num_list_representation, label)
        loss.backward()
        opt.step()
        loss_sum += loss.item()
    print(f'epoch {e} loss {loss_sum/len(trainloader):.4f}')
    if (e+1) % 2 == 0: #launch eval
        model.eval()
        err = 0
        num = 0
        for i,(x, y) in enumerate(evalset):
            x = x.to(device)
            y_pred = model.generate(x[None,...])
            hyp = ' '.join([str(i) for i in y_pred[1:-1]])
            y = y.numpy().tolist()
            # find the first 22 in list y
            y = y[:y.index(22)]
            ref = ' '.join([str(i) for i in y[1:]])


            err += editdistance.eval(hyp, ref)
            num += len(ref.split())

        print(f'error rate {err/num:.2%},  ({err}/{num})')
        if err/num < min_err:
            torch.save(model.state_dict(), f'{drive_path}model_mix.pt')
            min_err = err/num
        model.train()


weights loaded
epoch 0 loss 0.0309
epoch 1 loss 0.0292


<ipython-input-22-4c1151b4c32e>:133: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.nn.functional.pad(torch.tensor(x), (0, self.seq_len-len(x)), value=23)
<ipython-input-22-4c1151b4c32e>:108: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs = torch.nn.functional.softmax(logits[-1])


error rate 52.50%,  (2666/5078)
epoch 2 loss 0.0253
epoch 3 loss 0.0245
error rate 46.85%,  (2379/5078)
epoch 4 loss 0.0204
epoch 5 loss 0.0197
error rate 43.28%,  (2198/5078)
epoch 6 loss 0.0205
epoch 7 loss 0.0201
error rate 38.62%,  (1961/5078)
epoch 8 loss 0.0167
epoch 9 loss 0.0178
error rate 47.32%,  (2403/5078)
epoch 10 loss 0.0168
epoch 11 loss 0.0153
error rate 37.83%,  (1921/5078)
epoch 12 loss 0.0148
epoch 13 loss 0.0166
error rate 34.94%,  (1774/5078)
epoch 14 loss 0.0183
epoch 15 loss 0.0168
error rate 38.76%,  (1968/5078)
epoch 16 loss 0.0163
epoch 17 loss 0.0154
error rate 33.38%,  (1695/5078)
epoch 18 loss 0.0142
epoch 19 loss 0.0144
error rate 33.75%,  (1714/5078)
epoch 20 loss 0.0150


# Test the network

In [ ]:
if os.path.exists(drive_path+'model_mix.pt'):
    model.load_state_dict(torch.load(drive_path+'model_mix.pt'))
    print('weights loaded')
device = 'cuda'
model.to(device)

model.eval()
err = 0
num = 0
for i,(x, y) in enumerate(testset):
    x = x.to(device)
    y_pred = model.generate(x[None,...])
    hyp = ' '.join([str(i) for i in y_pred[1:-1]])
    y = y.numpy().tolist()
    # find the first 22 in list y
    y = y[:y.index(22)]
    ref = ' '.join([str(i) for i in y[1:]])

    print(f'sample {i}:\n\tpredicted: \t{hyp} \n\tlabel: \t\t{ref}')

    err += editdistance.eval(hyp, ref)
    num += len(ref.split())

print(f'error rate {err/num:.2%},  ({err}/{num})')

weights loaded
sample 0:
	predicted: 	3 19 3 21 3 
	label: 		3 8 3 19 1 1 18 3 21 1 4
sample 1:
	predicted: 	6 19 6 21 6 
	label: 		7 19 7 21 7


<ipython-input-7-4c1151b4c32e>:133: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.nn.functional.pad(torch.tensor(x), (0, self.seq_len-len(x)), value=23)
<ipython-input-7-4c1151b4c32e>:108: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs = torch.nn.functional.softmax(logits[-1])


Streaming output truncated to the last 5000 lines.
	label: 		9 0 5 7 19 9 18 1 2 21 2 1
sample 334:
	predicted: 	3 6 4 6 19 9 18 1 0 21 1 9 
	label: 		3 6 4 6 19 9 18 1 0 21 1 9
sample 335:
	predicted: 	6 6 19 1 2 21 1 2 
	label: 		6 0 19 6 21 6
sample 336:
	predicted: 	3 19 3 21 3 
	label: 		0 1 19 1 21 1
sample 337:
	predicted: 	8 1 4 19 9 18 4 21 1 3 
	label: 		8 1 4 19 9 18 4 21 1 3
sample 338:
	predicted: 	8 6 19 1 4 21 1 4 
	label: 		8 7 1 0 19 1 5 18 1 21 1 6
sample 339:
	predicted: 	3 4 19 7 21 7 
	label: 		3 4 9 7 19 7 18 1 6 21 2 3
sample 340:
	predicted: 	3 19 3 21 3 
	label: 		0 19 0 21 0
sample 341:
	predicted: 	3 3 2 0 19 6 18 2 21 8 
	label: 		3 0 2 0 19 3 18 2 21 5
sample 342:
	predicted: 	1 6 19 7 21 7 
	label: 		1 7 19 8 21 8
sample 343:
	predicted: 	6 19 6 21 6 
	label: 		7 7 2 5 19 1 4 18 7 21 2 1
sample 344:
	predicted: 	3 6 19 9 21 9 
	label: 		3 6 19 9 21 9
sample 345:
	predicted: 	3 6 6 19 9 18 6 21 1 5 
	label: 		9 7 6 19 1 6 18 6 21 2 2
sample 346:
	predicted: